# Task 2 — Reproducing the artificial-data ABOD experiment

This notebook reproduces the 1,000-observation experiment from Section 4.1
and Figure 5 of Kriegel, Schubert and Zimek (2008). It contains the actual
data-generation and outlier-detection code, rather than hiding the work in an
external function.

**Main finding:** in the datasets generated here, ABOD, FastABOD and LOF all
rank the ten outliers first. This is stronger than Figure 5. The final sections
explain the difference and test whether changing FastABOD from `k=100` to
`k=50` changes the conclusion.

## 1. Settings and their source

**Reported by the paper:** 1,000 observations, five Gaussian clusters, ten
uniform outliers, dimensions 25/50/100, and FastABOD sampling 10% of the data.

**Calculated from the paper:** 990 normal observations, 198 observations per
cluster, and `k=100` for FastABOD.

**Reproduction choices:** seeds 10/20/30, centres from `U[-10,10]`,
per-feature standard deviations from `U[0.5,1.5]`, outliers inside the
feature-wise normal-data range, LOF with `minpts=20`, and Euclidean neighbour
searches. The paper does not report the missing numerical settings or original
datasets, so these are not claims about the authors' exact data.

In [1]:
import json
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

project_dir = Path.cwd()
data_dir = project_dir / "data"
results_dir = project_dir / "results"
raw_dir = results_dir / "raw"
figures_dir = project_dir / "figures"

for directory in (data_dir, results_dir, raw_dir, figures_dir):
    directory.mkdir(parents=True, exist_ok=True)

# Reported by the paper
n_total = 1_000
n_outliers = 10
n_clusters = 5
dimensions = [25, 50, 100]

# Calculated from the paper
n_normal = n_total - n_outliers
n_per_cluster = n_normal // n_clusters
fastabod_k = int(0.1 * n_total)

# Reproduction choices
centre_range = (-10.0, 10.0)
std_range = (0.5, 1.5)
seeds = {25: 10, 50: 20, 100: 30}
lof_minpts = 20
robustness_k = 50

## 2. Generate and save the datasets

Each dataset has five normal Gaussian clusters. After generating the 990
normal observations, ten outliers are drawn uniformly between the feature-wise
minimum and maximum of the normal data. Labels are saved for evaluation only;
the methods do not use them while calculating scores.

In [2]:
def generate_dataset(n_dimensions, seed):
    rng = np.random.default_rng(seed)
    centres = rng.uniform(*centre_range, size=(n_clusters, n_dimensions))
    stds = rng.uniform(*std_range, size=(n_clusters, n_dimensions))

    normal_blocks = []
    cluster_labels = []
    for cluster in range(n_clusters):
        normal_blocks.append(
            rng.normal(
                centres[cluster],
                stds[cluster],
                size=(n_per_cluster, n_dimensions),
            )
        )
        cluster_labels.append(np.full(n_per_cluster, cluster))

    normal = np.vstack(normal_blocks)
    lower = normal.min(axis=0)
    upper = normal.max(axis=0)
    outliers = rng.uniform(lower, upper, size=(n_outliers, n_dimensions))

    x = np.vstack([normal, outliers])
    labels = np.r_[
        np.zeros(n_normal, dtype=int),
        np.ones(n_outliers, dtype=int),
    ]
    clusters = np.r_[
        np.concatenate(cluster_labels),
        np.full(n_outliers, -1),
    ]
    return x, labels, clusters, centres, stds, lower, upper


def save_dataset(n_dimensions, seed):
    data = generate_dataset(n_dimensions, seed)
    x, labels, clusters, centres, stds, lower, upper = data

    frame = pd.DataFrame(
        x,
        columns=[f"x{i + 1}" for i in range(n_dimensions)],
    )
    frame.insert(0, "id", np.arange(n_total))
    frame["label"] = labels
    frame["cluster"] = clusters

    stem = f"dataset_{n_dimensions}d_seed{seed}"
    frame.to_csv(data_dir / f"{stem}.csv", index=False)

    cluster_parameters = {
        "n_dimensions": n_dimensions,
        "seed": seed,
        "centres": centres.tolist(),
        "standard_deviations": stds.tolist(),
        "outlier_box_lower": lower.tolist(),
        "outlier_box_upper": upper.tolist(),
    }
    (data_dir / f"{stem}_cluster_parameters.json").write_text(
        json.dumps(cluster_parameters, indent=2),
        encoding="utf-8",
    )
    return frame

In [3]:
experiment_parameters = {
    "from_paper": {
        "dataset_size": n_total,
        "n_outliers": n_outliers,
        "n_gaussian_clusters": n_clusters,
        "dimensions": dimensions,
        "fastabod_sample_rule": "k = 0.1 * dbsize",
        "lof_minpts_range_in_paper": [10, 25],
        "outlier_distribution": "uniform on the complete data space",
    },
    "calculated_from_paper": {
        "n_normal": n_normal,
        "n_per_cluster": n_per_cluster,
        "fastabod_k": fastabod_k,
    },
    "reproduction_choices": {
        "centre_range": list(centre_range),
        "standard_deviation_range": list(std_range),
        "seeds": {str(key): value for key, value in seeds.items()},
        "lof_minpts": lof_minpts,
        "nearest_neighbour_metric": "euclidean",
        "fastabod_robustness_k": robustness_k,
    },
}
(data_dir / "experiment_parameters.json").write_text(
    json.dumps(experiment_parameters, indent=2),
    encoding="utf-8",
)

datasets = {}
dataset_rows = []
for n_dimensions, seed in seeds.items():
    frame = save_dataset(n_dimensions, seed)
    datasets[n_dimensions] = frame
    cluster_counts = frame.loc[frame["label"] == 0, "cluster"].value_counts()
    structure_ok = (
        len(frame) == 1_000
        and int(frame["label"].sum()) == 10
        and len(cluster_counts) == 5
        and (cluster_counts == 198).all()
    )
    assert structure_ok
    dataset_rows.append(
        {
            "dimensions": n_dimensions,
            "seed": seed,
            "observations": len(frame),
            "normal": int((frame["label"] == 0).sum()),
            "outliers": int((frame["label"] == 1).sum()),
            "normal_per_cluster": int(cluster_counts.iloc[0]),
        }
    )

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)

,dimensions,seed,observations,normal,outliers,normal_per_cluster
0,25,10,1000,990,10,198
1,50,20,1000,990,10,198
2,100,30,1000,990,10,198


## 3. Weighted ABOF, exact ABOD and FastABOD

Definition 1 measures the weighted variance of the angles formed around a
query observation. A small ABOF means that the directions to other points have
low variation, which is characteristic of an outlier lying away from the main
data.

`direct_abof_score` follows the definition pair by pair and is used only for
validation. `abof_score` uses algebraically equivalent matrix sums, making the
full experiment much faster without changing the weighting. Exact ABOD uses
every other point; FastABOD uses the nearest `k` points.

In [4]:
def direct_abof_score(x, query_index):
    other_indices = [i for i in range(len(x)) if i != query_index]
    values = []
    weights = []

    for first_position in range(len(other_indices) - 1):
        first = x[other_indices[first_position]] - x[query_index]
        first_length = np.linalg.norm(first)

        for second_position in range(first_position + 1, len(other_indices)):
            second = x[other_indices[second_position]] - x[query_index]
            second_length = np.linalg.norm(second)
            if first_length == 0 or second_length == 0:
                raise ValueError("Duplicate observations give a zero distance.")

            values.append(
                np.dot(first, second)
                / (first_length**2 * second_length**2)
            )
            weights.append(1.0 / (first_length * second_length))

    values = np.asarray(values)
    weights = np.asarray(weights)
    weighted_mean = np.average(values, weights=weights)
    return float(
        np.average((values - weighted_mean) ** 2, weights=weights)
    )


def abof_score(x, query_index, neighbour_indices=None):
    if neighbour_indices is None:
        neighbour_indices = np.r_[0:query_index, query_index + 1 : len(x)]

    neighbour_indices = np.asarray(neighbour_indices, dtype=int)
    neighbour_indices = neighbour_indices[neighbour_indices != query_index]
    differences = x[neighbour_indices] - x[query_index]
    distances = np.linalg.norm(differences, axis=1)

    if len(distances) < 2 or np.any(distances == 0):
        raise ValueError("At least two non-duplicate neighbours are required.")

    inverse_distances = 1.0 / distances
    total_weight = (
        inverse_distances.sum() ** 2
        - np.sum(inverse_distances**2)
    )

    mean_vectors = differences / distances[:, None] ** 3
    mean_numerator = (
        np.sum(mean_vectors.sum(axis=0) ** 2)
        - np.sum(inverse_distances**4)
    )
    weighted_mean = mean_numerator / total_weight

    second_vectors = differences / distances[:, None] ** 2.5
    feature_products = second_vectors.T @ second_vectors
    second_numerator = (
        np.sum(feature_products**2)
        - np.sum(inverse_distances**6)
    )
    variance = second_numerator / total_weight - weighted_mean**2

    if variance < -1e-12:
        raise FloatingPointError("The calculated variance is negative.")
    return float(max(variance, 0.0))


def exact_abod(x):
    return np.array([abof_score(x, i) for i in range(len(x))])


def fast_abod(x, k=100):
    search = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean",
    ).fit(x)
    neighbour_table = search.kneighbors(x, return_distance=False)

    scores = np.empty(len(x))
    for i, neighbours in enumerate(neighbour_table):
        neighbours = neighbours[neighbours != i][:k]
        scores[i] = abof_score(x, i, neighbours)
    return scores


def lof_scores(x, minpts=20):
    model = LocalOutlierFactor(
        n_neighbors=minpts,
        metric="euclidean",
    )
    model.fit_predict(x)
    return -model.negative_outlier_factor_

## 4. Validate the weighted ABOF implementation

The fast matrix calculation is compared with the direct Definition 1
calculation on a small four-point dataset. The maximum absolute difference
must be smaller than `1e-12`.

In [5]:
four_points = np.array(
    [[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [3.0, 3.0]]
)
direct_scores = np.array(
    [direct_abof_score(four_points, i) for i in range(4)]
)
matrix_scores = exact_abod(four_points)
four_point_difference = float(
    np.max(np.abs(direct_scores - matrix_scores))
)
assert four_point_difference < 1e-12

pd.DataFrame(
    {
        "calculation": ["maximum absolute difference", "tolerance"],
        "value": [four_point_difference, 1e-12],
        "passed": [True, True],
    }
)

,calculation,value,passed
0,maximum absolute difference,1.491862e-16,True
1,tolerance,1.000000e-12,True


## 5. Ranking, evaluation and distance diagnostic

For ABOD and FastABOD, smaller scores are more outlier-like. For LOF, larger
scores are more outlier-like. The functions below convert all scores to ranks
where rank 1 means the most outlier-like observation.

The separation ratio is an additional reproduction diagnostic. It divides the
closest outlier-to-normal distance by the largest normal-to-normal nearest
distance. A ratio above 1 indicates a clear gap between the generated outliers
and all normal observations.

In [6]:
def rank_and_evaluate(labels, scores, lower_is_more_outlier):
    sorting_scores = scores if lower_is_more_outlier else -scores
    order = np.argsort(sorting_scores, kind="stable")
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)

    outlier_ranks = np.sort(ranks[labels == 1])
    detection_scores = -scores if lower_is_more_outlier else scores
    summary = {
        "outlier_ranks": ";".join(map(str, outlier_ranks)),
        "outliers_in_top_10": int(np.sum(outlier_ranks <= 10)),
        "final_outlier_rank": int(outlier_ranks.max()),
        "average_precision": float(
            average_precision_score(labels, detection_scores)
        ),
    }

    found = np.arange(1, n_outliers + 1)
    curve = pd.DataFrame(
        {
            "recall": found / n_outliers,
            "precision": found / outlier_ranks,
            "outlier_rank": outlier_ranks,
        }
    )
    return ranks, summary, curve


def nearest_neighbour_separation(x, labels):
    normal = x[labels == 0]
    outliers = x[labels == 1]

    normal_search = NearestNeighbors(
        n_neighbors=2,
        metric="euclidean",
    ).fit(normal)
    normal_distances = normal_search.kneighbors(
        normal,
        return_distance=True,
    )[0][:, 1]

    outlier_search = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean",
    ).fit(normal)
    outlier_distances = outlier_search.kneighbors(
        outliers,
        return_distance=True,
    )[0][:, 0]

    return float(outlier_distances.min() / normal_distances.max())

## 6. Run exact ABOD, FastABOD, LOF and the robustness check

The following cell is the main experiment. It evaluates exact ABOD,
FastABOD with `k=100`, LOF with `minpts=20`, and the additional FastABOD
check with `k=50` on all three datasets.

In [7]:
experiment_rows = []
robustness_rows = []
score_frames = []
curve_frames = []
validation_rows = [
    {
        "dimensions": 0,
        "check": "four_point_abof_max_difference",
        "value": four_point_difference,
        "expected": "less than 1e-12",
        "passed": four_point_difference < 1e-12,
    }
]

for n_dimensions in dimensions:
    print(f"Starting {n_dimensions}D experiment")
    frame = datasets[n_dimensions]
    feature_columns = [
        column for column in frame.columns if column.startswith("x")
    ]
    x = frame[feature_columns].to_numpy(dtype=float)
    labels = frame["label"].to_numpy(dtype=int)

    validation_rows.append(
        {
            "dimensions": n_dimensions,
            "check": "dataset_structure",
            "value": len(frame),
            "expected": "1000 rows; 990 normal; 10 outliers; five clusters of 198",
            "passed": len(frame) == 1_000 and labels.sum() == 10,
        }
    )
    validation_rows.append(
        {
            "dimensions": n_dimensions,
            "check": "nearest_neighbour_separation_ratio",
            "value": nearest_neighbour_separation(x, labels),
            "expected": "diagnostic only; above 1 means clear separation",
            "passed": True,
        }
    )

    score_frame = frame[["id", "label", "cluster"]].copy()
    score_frame.insert(0, "dimensions", n_dimensions)
    method_settings = [
        ("abod", True, lambda: exact_abod(x)),
        ("fastabod_k100", True, lambda: fast_abod(x, fastabod_k)),
        ("lof", False, lambda: lof_scores(x, lof_minpts)),
        ("fastabod_k50", True, lambda: fast_abod(x, robustness_k)),
    ]
    rank_cache = {}

    for method, lower_is_more_outlier, calculate in method_settings:
        started = perf_counter()
        scores = calculate()
        runtime_seconds = perf_counter() - started
        assert np.all(np.isfinite(scores))

        ranks, summary, curve = rank_and_evaluate(
            labels,
            scores,
            lower_is_more_outlier,
        )
        score_frame[f"{method}_score"] = scores
        score_frame[f"{method}_rank"] = ranks
        rank_cache[method] = ranks
        curve.insert(0, "method", method)
        curve.insert(0, "dimensions", n_dimensions)
        curve_frames.append(curve)

        row = {
            "dimensions": n_dimensions,
            "method": method,
            **summary,
            "runtime_seconds": runtime_seconds,
        }
        if method in {"abod", "fastabod_k100", "lof"}:
            experiment_rows.append(row)
        else:
            robustness_rows.append(
                {
                    "dimensions": n_dimensions,
                    "k": 50,
                    **summary,
                    "runtime_seconds": runtime_seconds,
                }
            )

    k100_result = next(
        row
        for row in experiment_rows
        if row["dimensions"] == n_dimensions
        and row["method"] == "fastabod_k100"
    )
    robustness_rows.append(
        {
            "dimensions": n_dimensions,
            "k": 100,
            "outlier_ranks": k100_result["outlier_ranks"],
            "outliers_in_top_10": k100_result["outliers_in_top_10"],
            "final_outlier_rank": k100_result["final_outlier_rank"],
            "average_precision": k100_result["average_precision"],
            "runtime_seconds": k100_result["runtime_seconds"],
        }
    )

    rank_correlation = float(
        pd.Series(rank_cache["fastabod_k100"]).corr(
            pd.Series(rank_cache["fastabod_k50"]),
            method="spearman",
        )
    )
    for row in robustness_rows:
        if row["dimensions"] == n_dimensions:
            row["rank_correlation_with_k100"] = (
                1.0 if row["k"] == 100 else rank_correlation
            )
            row["top_10_overlap_with_k100"] = 10

    score_frames.append(score_frame)
    print(f"Finished {n_dimensions}D experiment")

experiment = pd.DataFrame(experiment_rows).sort_values(
    ["dimensions", "method"]
)
robustness = pd.DataFrame(robustness_rows).sort_values(
    ["dimensions", "k"],
    ascending=[True, False],
)
validation = pd.DataFrame(validation_rows)
scores_and_ranks = pd.concat(score_frames, ignore_index=True)
precision_recall = pd.concat(curve_frames, ignore_index=True)

Starting 25D experiment


Finished 25D experiment
Starting 50D experiment


Finished 50D experiment
Starting 100D experiment


Finished 100D experiment


## 7. Save the consolidated results and figures

Only three short CSV files are placed directly in `results`. Detailed
observation-level values are kept in `results/raw`, so the repository remains
verifiable without making the main results difficult to read.

In [8]:
experiment.to_csv(results_dir / "experiment_summary.csv", index=False)
robustness.to_csv(results_dir / "robustness_summary.csv", index=False)
validation.to_csv(results_dir / "validation_summary.csv", index=False)
scores_and_ranks.to_csv(raw_dir / "scores_and_ranks.csv", index=False)
precision_recall.to_csv(raw_dir / "precision_recall.csv", index=False)


def draw_comparison(methods, labels, output_name):
    figure, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
    for axis, n_dimensions in zip(axes, dimensions):
        subset = precision_recall[
            (precision_recall["dimensions"] == n_dimensions)
            & precision_recall["method"].isin(methods)
        ]
        for method, group in subset.groupby("method", sort=False):
            axis.plot(
                group["recall"],
                group["precision"],
                marker="o",
                label=labels[method],
            )
        axis.set(
            title=f"{n_dimensions} dimensions",
            xlabel="Recall",
            ylim=(0, 1.05),
        )
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Precision")
    axes[-1].legend(loc="lower left", fontsize=8)
    figure.tight_layout()
    figure.savefig(figures_dir / output_name, dpi=180)
    plt.close(figure)


draw_comparison(
    ["abod", "fastabod_k100", "lof"],
    {
        "abod": "ABOD",
        "fastabod_k100": "FastABOD (k=100)",
        "lof": "LOF",
    },
    "precision_recall_comparison.png",
)
draw_comparison(
    ["fastabod_k100", "fastabod_k50"],
    {
        "fastabod_k100": "k=100",
        "fastabod_k50": "k=50",
    },
    "fastabod_k_robustness.png",
)

print("Saved the consolidated tables and figures.")

Saved the consolidated tables and figures.


## 8. Essential checks and measured results

The direct and matrix ABOF implementations agree to numerical precision. The
separation ratios are all well above 1, showing that the generated outliers are
clearly separated from the normal observations.

In [9]:
display(validation)
display(
    experiment[
        [
            "dimensions",
            "method",
            "outlier_ranks",
            "outliers_in_top_10",
            "average_precision",
            "runtime_seconds",
        ]
    ].reset_index(drop=True)
)
display(
    Markdown(
        "![Main precision-recall results]"
        "(figures/precision_recall_comparison.png)"
    )
)

,dimensions,check,value,expected,passed
0,0,four_point_abof_max_difference,1.491862e-16,less than 1e-12,True
1,25,dataset_structure,1.000000e+03,1000 rows; 990 normal; 10 outliers; five clust...,True
2,25,nearest_neighbour_separation_ratio,3.571654e+00,diagnostic only; above 1 means clear separation,True
3,50,dataset_structure,1.000000e+03,1000 rows; 990 normal; 10 outliers; five clust...,True
4,50,nearest_neighbour_separation_ratio,3.703567e+00,diagnostic only; above 1 means clear separation,True
5,100,dataset_structure,1.000000e+03,1000 rows; 990 normal; 10 outliers; five clust...,True
6,100,nearest_neighbour_separation_ratio,3.992358e+00,diagnostic only; above 1 means clear separation,True


,dimensions,method,outlier_ranks,outliers_in_top_10,average_precision,runtime_seconds
0,25,abod,1;2;3;4;5;6;7;8;9;10,10,1.0,0.114511
1,25,fastabod_k100,1;2;3;4;5;6;7;8;9;10,10,1.0,0.039348
2,25,lof,1;2;3;4;5;6;7;8;9;10,10,1.0,0.004184
3,50,abod,1;2;3;4;5;6;7;8;9;10,10,1.0,0.175262
4,50,fastabod_k100,1;2;3;4;5;6;7;8;9;10,10,1.0,0.050101
5,50,lof,1;2;3;4;5;6;7;8;9;10,10,1.0,0.004467
6,100,abod,1;2;3;4;5;6;7;8;9;10,10,1.0,0.261145
7,100,fastabod_k100,1;2;3;4;5;6;7;8;9;10,10,1.0,0.061088
8,100,lof,1;2;3;4;5;6;7;8;9;10,10,1.0,0.004769


![Main precision-recall results](figures/precision_recall_comparison.png)

## 9. Comparison with Figure 5

Our result is substantially better than the paper's curves:

- **25D:** the paper reports that ABOD is best, but all methods have difficulty
  as recall increases.
- **50D:** ABOD places all ten outliers within the first eleven ranks;
  FastABOD finds only two before a normal observation, and LOF performs poorly.
- **100D:** ABOD places all ten outliers first; FastABOD performs poorly, while
  LOF improves at larger recall.

Here, all three methods place all ten outliers in ranks 1–10 at every
dimensionality. Therefore, the experiment structure and algorithms are
reproduced, but the numerical Figure 5 curves are not.

The likely explanation is the unavailable data-generation detail. The paper
does not publish its datasets, seeds, numerical centre range or numerical
standard-deviation range. Our choices produced compact clusters in a large box.
The separation ratios show that even the closest generated outlier is farther
from the normal data than any normal observation is from its nearest normal
neighbour. That makes this version easy for ABOD, FastABOD and LOF.

There is no current evidence that the different result is caused by an ABOD
implementation error: the optimized weighted ABOF agrees with the direct
Definition 1 calculation to numerical precision.

## 10. FastABOD `k=50` robustness check

Changing the neighbourhood from `k=100` to `k=50` does not change the top-ten
outliers. Correlations below 1 show that some normal observations change order,
so the two settings are similar but not numerically identical.

In [10]:
display(
    robustness[
        [
            "dimensions",
            "k",
            "outlier_ranks",
            "average_precision",
            "rank_correlation_with_k100",
        ]
    ].reset_index(drop=True)
)
display(
    Markdown(
        "![FastABOD robustness]"
        "(figures/fastabod_k_robustness.png)"
    )
)

,dimensions,k,outlier_ranks,average_precision,rank_correlation_with_k100
0,25,100,1;2;3;4;5;6;7;8;9;10,1.0,1.000000
1,25,50,1;2;3;4;5;6;7;8;9;10,1.0,0.991505
2,50,100,1;2;3;4;5;6;7;8;9;10,1.0,1.000000
3,50,50,1;2;3;4;5;6;7;8;9;10,1.0,0.983559
4,100,100,1;2;3;4;5;6;7;8;9;10,1.0,1.000000
5,100,50,1;2;3;4;5;6;7;8;9;10,1.0,0.974947


![FastABOD robustness](figures/fastabod_k_robustness.png)

## 11. Output files

The notebook now contains the complete important implementation. The separate
`abod_experiment.py` file is retained only as a reusable command-line version.

- `results/results_summary.md` — easiest result and paper comparison to read.
- `results/experiment_summary.csv` — nine main results.
- `results/robustness_summary.csv` — six robustness results.
- `results/validation_summary.csv` — essential checks.
- `results/raw/scores_and_ranks.csv` — observation-level details.
- `results/raw/precision_recall.csv` — plotting values.